In the previous session we trained a model for predicting churn and evaluated it. Now let's deploy it

In [11]:
# pandas는 표(테이블) 데이터를 다루는 라이브러리로, 고객 이탈 CSV를 DataFrame으로 다루기 위해 쓰인다.
import pandas as pd
# numpy는 숫자 배열 연산을 다루는 라이브러리로, 교차검증 평균·표준편차를 계산하기 위해 쓰인다.
import numpy as np

# train_test_split()은 데이터를 훈련/검증/테스트 비율로 나눠 주는 함수로,
# 모델이 보지 못한 데이터로 성능을 공정하게 평가하기 위해 쓰인다.
from sklearn.model_selection import train_test_split
# KFold는 데이터를 K개 폴드로 나눠 교차검증하는 도구로,
# 한 번 나눈 검증셋에만 의존하지 않고 성능을 더 안정적으로 추정하기 위해 쓰인다.
from sklearn.model_selection import KFold

# DictVectorizer는 딕셔너리 특성을 숫자 행렬로 바꿔 주는 변환기로,
# 범주형 변수를 원-핫 인코딩하기 위해 쓰인다.
from sklearn.feature_extraction import DictVectorizer
# LogisticRegression은 이진 분류용 로지스틱 회귀 모델로,
# 고객이 이탈할지 여부를 예측하기 위해 쓰인다.
from sklearn.linear_model import LogisticRegression
# roc_auc_score()는 ROC AUC를 한 번에 구해 주는 함수로,
# 배포 전에 모델 품질을 숫자 하나로 확인하기 위해 쓰인다.
from sklearn.metrics import roc_auc_score

# set_config는 sklearn이 모델 객체를 화면에 어떻게 보여줄지 바꾸는 함수다.
from sklearn import set_config
# display="text": 모델 미리보기를 HTML이 아니라 글자로만 보여 준다.
# 한국어 Windows(cp949)에서 sklearn HTML 다이어그램이 깨지는 것을 막기 위해 필요하다.
set_config(display="text")

# warnings는 파이썬 경고를 켜고 끄는 모듈이다.
import warnings
# ConvergenceWarning은 로지스틱 회귀가 반복 한도 안에 최적점을 못 찾았을 때 나는 경고다.
from sklearn.exceptions import ConvergenceWarning
# 이 데이터(요금·계약기간 스케일이 다름)에서는 기본 lbfgs가 한도 안에 수렴하지 못한다.
# 경고만 숨긴다. fit은 끝나고 예측은 나온다.
warnings.filterwarnings("ignore", category=ConvergenceWarning)


In [12]:
# read_csv()는 CSV 파일을 DataFrame으로 읽어 주는 함수로, 3·4주차와 같은 통신사 이탈 데이터를 불러오기 위해 쓰인다.
df = pd.read_csv('data-week-3.csv')

# columns는 컬럼 이름 목록이고, str.lower()/replace()는 문자열을 소문자로 바꾸고 공백을 _ 로 바꿔 주는 메서드로,
# 컬럼명을 다루기 쉽게 통일하기 위해 쓰인다.
df.columns = df.columns.str.lower().str.replace(' ', '_')

# select_dtypes()는 지정한 자료형 컬럼만 골라 주는 메서드이며,
# pandas 3 이상에서는 텍스트 데이터의 기본 타입이 string으로 변경되었기 때문에
# 기존 데이터(object)와 최신 string 타입을 모두 포함하여 텍스트형(범주형) 컬럼들을 누락 없이 안전하게 선택하기 위해 쓰인다.
# select_dtypes()가 반환한 필터링된 데이터프레임에서 컬럼 이름 목록(Index)만 뽑아내기 위해 .columns를 붙인다.
categorical_columns = list(df.select_dtypes(include=['object', 'string']).columns)

# 범주형 컬럼마다 값을 소문자로 바꾸고 공백을 _ 로 정규화한다.
for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

# to_numeric()은 문자열을 숫자로 바꿔 주는 함수이자 totalcharges의 빈 칸을 숫자 결측으로 만들기 위해 쓰인다.
df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
# fillna()는 결측값을 채워 주는 메서드로, 비어 있는 총요금을 0으로 두기 위해 쓰인다.
df.totalcharges = df.totalcharges.fillna(0)

# (df.churn == 'yes')는 이탈 여부를 True/False로 만들고 astype(int)는 1/0으로 바꿔 주는 메서드로,
# 분류 모델의 정답 레이블을 숫자로 만들기 위해 쓰인다.
df.churn = (df.churn == 'yes').astype(int)


In [13]:
# train_test_split()은 데이터를 비율대로 나눠 주는 함수로,
# 전체의 20%를 테스트셋으로 떼어 최종 평가·배포 확인용으로 쓰기 위해 쓰인다.
# random_state=1은 다시 실행해도 같은 분할이 나오게 한다.
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)


In [14]:
# 수치형 특성 목록. 계약 기간·월요금·총요금은 숫자 그대로 모델에 넣는다.
numerical = ['tenure', 'monthlycharges', 'totalcharges']

# 범주형 특성 목록. 성별·계약 종류처럼 문자(또는 0/1) 값은 원-핫 인코딩 대상이다.
categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]


In [15]:
# train()은 데이터프레임과 정답을 적용해 벡터라이저+로지스틱 모델을 학습해 돌려 주는 함수로,
# 교차검증에서 같은 학습 절차를 반복하기 위해 묶어둠.
def train(df_train, y_train, C=1.0):
    # to_dict()은 행들을 {컬럼명: 값} 딕셔너리 리스트로 바꿔 주는 메서드로,
    # DictVectorizer 입력 형태로 가공하기 위해 쓰인다.
    dicts = df_train[categorical + numerical].to_dict(orient='records')

    # DictVectorizer()은 딕셔너리를 수치 행렬로 변환해 주는 변환기다.
    # sparse=False는 희소 행렬이 아니라 밀집 NumPy 배열로 받기 위해 쓰인다.
    dv = DictVectorizer(sparse=False)
    # fit_transform()은 카테고리 고유 값을 학습하고 숫자로 바꿔 주는 메서드로,
    # 학습용 피처 행렬(X_train)을 만들기 위해 쓰인다.
    X_train = dv.fit_transform(dicts)

    # C는 규제 강도(작을수록 규제가 큼). max_iter=1000은 레슨 원본과 같다.
    model = LogisticRegression(C=C, max_iter=1000)
    # fit()은 피처와 정답으로 가중치를 학습하는 메서드다.
    model.fit(X_train, y_train)

    return dv, model


In [16]:
# predict()는 이미 학습된 dv와 model로 이탈 확률만 계산해 주는 함수로,
# 검증/테스트셋에 같은 전처리를 적용하기 위해 쓰인다.
def predict(df, dv, model):
    dicts = df[categorical + numerical].to_dict(orient='records')

    # transform만 사용한다(fit은 훈련셋에서 이미 끝남). 새 고객에도 같은 컬럼 순서를 맞춘다.
    X = dv.transform(dicts)
    # predict_proba()는 클래스별 확률을 2차원 배열로 돌려 주는 메서드다.
    # [:, 1]은 이탈(클래스 1) 확률만 골라 배포 API에서 쓸 점수를 만든다.
    y_pred = model.predict_proba(X)[:, 1]

    return y_pred


In [17]:
# C는 로지스틱 회귀의 규제 강도다. 1.0은 기본값에 가깝고 레슨에서 최종 모델로 쓴다.
C = 1.0
# n_splits는 교차검증 폴드 개수다. 5면 데이터를 다섯 번 나눠 평균 성능을 본다.
n_splits = 5


In [18]:
# KFold()는 인덱스를 K개로 나눠 주는 객체다.
# shuffle=True는 앞에서부터 자르지 않고 섞은 뒤 나눠, 한쪽으로 치우친 폴드를 줄이기 위해 쓰인다.
kfold = KFold(n_splits=n_splits, shuffle=True, random_state=1)


# 폴드마다 나온 AUC(모델의 평균 성능)를 모아 평균·표준편차를 내기 위한 리스트다.
scores = []

# split()은 훈련/검증 인덱스 쌍을 반복해 주는 메서드로, 5번 학습·평가를 돌리기 위해 쓰인다.
for train_idx, val_idx in kfold.split(df_full_train):
    # iloc[]은 정수 위치로 행을 골라 주는 인덱서다. 이번 폴드의 훈련·검증 표를 만든다.
    df_train = df_full_train.iloc[train_idx]
    df_val = df_full_train.iloc[val_idx]

    # .values는 시리즈를 NumPy 배열로 바꿔 주는 속성으로, sklearn이 받을 정답 벡터를 만들기 위해 쓰인다.
    y_train = df_train.churn.values
    y_val = df_val.churn.values

    dv, model = train(df_train, y_train, C=C)
    y_pred = predict(df_val, dv, model)

    # roc_auc_score()로 이번 폴드 검증 AUC(모델의 평균 성능)를 구하고 리스트에 쌓는다.
    auc = roc_auc_score(y_val, y_pred)
    scores.append(auc)

# mean()/std()는 평균과 표준편차를 구해 주는 함수로, 한 폴드에 휘둘리지 않는 성능을 보고하기 위해 쓰인다.
print('C=%s %.3f +- %.3f' % (C, np.mean(scores), np.std(scores)))


C=1.0 0.842 +- 0.007


In [20]:
# 폴드별 AUC 원값을 확인한다. 평균만 보면 가려지는 폴드 간 차이를 보기 위해 쓰인다.
scores


[0.8446279564490428,
 0.8451633061846113,
 0.8334256472004816,
 0.8347728944170877,
 0.8516323617081358]

In [21]:
# 교차검증이 끝난 뒤, 훈련+검증을 합친 full_train 전체로 최종 모델을 학습한다.
# 배포에 넣을 가중치는 가능한 한 많은 데이터로 한 번 더 맞춘다.
dv, model = train(df_full_train, df_full_train.churn.values, C=1.0)
# 한 번도 학습에 안 쓴 테스트셋으로 이탈 확률을 구한다.
y_pred = predict(df_test, dv, model)

y_test = df_test.churn.values
# 테스트 AUC. 배포 전 "이 모델이 새 고객에도 쓸 만한지"를 확인하기 위해 쓰인다.
auc = roc_auc_score(y_test, y_pred)
auc


0.8583327916626041

Save the model

In [22]:
# pickle은 파이썬 객체를 파일로 저장·불러오는 라이브러리다.
# 학습이 끝난 DictVectorizer와 모델을 나중에 Flask에서 쓰기 위해 직렬화한다.
# Flask는 파이썬으로 가벼운 웹 서버(API)를 구축해 머신러닝 모델을 외부에서 호출할 수 있게 해주는 웹 프레임워크다.
import pickle

In [23]:
# f-string으로 규제값 C를 파일 이름에 넣는다. 어떤 설정으로 저장했는지 구분하기 위해 쓰인다.
output_file = f'model_C={C}.bin'


In [24]:
# 저장할 파일 이름을 확인한다.
output_file


'model_C=1.0.bin'

In [25]:
# open(..., 'wb')는 바이너리 쓰기 모드로 파일을 여는 함수다. pickle은 글자가 아니라 바이트를 쓴다.
f_out = open(output_file, 'wb')
# dump()는 객체를 파일에 쓰는 함수다. (dv, model) 튜플로 전처리기와 모델을 함께 저장한다.
pickle.dump((dv, model), f_out)
# close()는 파일을 닫아 버퍼를 디스크에 반영한다.
f_out.close()


In [26]:
# !ls -lh 는 셸에서 파일 크기까지 목록을 보여 주는 명령이다.
# 방금 만든 .bin이 생겼는지 확인하기 위해 쓰인다. Git Bash에서는 동작하고, cmd에서는 dir을 쓰면 된다.
!ls -lh *.bin


-rw-r--r-- 1 HP EliteBook 197121 2.5K  8�썡 30 19:29 model_C=1.0.bin


In [27]:
# with open()은 블록이 끝나면 파일을 자동으로 닫아 주는 구문이다.
# close()를 빼먹을 일을 줄이려고, 바로 위 저장을 같은 방식으로 한 번 더 보여 준다.
with open(output_file, 'wb') as f_out:
    pickle.dump((dv, model), f_out)


Load the model

In [28]:
# 아래 Load 구간을 노트북 위와 따로 실행할 수 있게 pickle을 다시 불러 온다.
import pickle


In [29]:
# 읽어 올 모델 파일 이름. 방금 저장한 model_C=1.0.bin과 같아야 한다.
input_file = 'model_C=1.0.bin'


In [30]:
# 'rb'는 바이너리 읽기 모드다. dump의 반대인 load()로 (dv, model)을 복원한다.
with open(input_file, 'rb') as f_in:
    dv, model = pickle.load(f_in)


In [32]:
# 불러온 모델 객체를 확인한다. max_iter 등이 학습 때와 같은지 보기 위해 쓰인다.
model


LogisticRegression(max_iter=1000)

In [33]:
dv

DictVectorizer(sparse=False)

In [34]:
# 한 명의 고객을 딕셔너리로 만든 예시다.
# API에 JSON으로 들어올 요청 몸체와 같은 모양이라, 배포 전 로컬에서 예측을 시험하기 위해 쓰인다.
customer = {
    'gender': 'female',
    'seniorcitizen': 0,
    'partner': 'yes',
    'dependents': 'no',
    'phoneservice': 'no',
    'multiplelines': 'no_phone_service',
    'internetservice': 'dsl',
    'onlinesecurity': 'no',
    'onlinebackup': 'yes',
    'deviceprotection': 'no',
    'techsupport': 'no',
    'streamingtv': 'no',
    'streamingmovies': 'no',
    'contract': 'month-to-month',
    'paperlessbilling': 'yes',
    'paymentmethod': 'electronic_check',
    'tenure': 1,
    'monthlycharges': 29.85,
    'totalcharges': 29.85
}


In [37]:
# transform()에 리스트로 한 명을 넣는다. 학습 때와 같은 컬럼으로 숫자 행렬 한 줄을 만든다.
X = dv.transform([customer])


In [38]:
# predict_proba()[0, 1]은 첫 번째 행의 이탈 확률이다.
# [0, 1]은 2차원 배열에서 첫 번째 고객([0])의 이탈할 확률(클래스 1, [1])을 골라내기 위한 인덱싱이다.
# 서비스가 고객 한 명에 대해 돌려 줄 숫자와 같다.
y_pred = model.predict_proba(X)[0, 1]


In [39]:
# print()로 입력 고객과 예측 확률을 함께 보여, 로컬 추론이 됐는지 확인한다.
print('input:', customer)
print('output:', y_pred)


input: {'gender': 'female', 'seniorcitizen': 0, 'partner': 'yes', 'dependents': 'no', 'phoneservice': 'no', 'multiplelines': 'no_phone_service', 'internetservice': 'dsl', 'onlinesecurity': 'no', 'onlinebackup': 'yes', 'deviceprotection': 'no', 'techsupport': 'no', 'streamingtv': 'no', 'streamingmovies': 'no', 'contract': 'month-to-month', 'paperlessbilling': 'yes', 'paymentmethod': 'electronic_check', 'tenure': 1, 'monthlycharges': 29.85, 'totalcharges': 29.85}
output: 0.627656987858


Making requests

In [44]:
# requests는 HTTP 요청을 보내는 라이브러리다.
# 나중에 띄운 Flask /predict 엔드포인트를 노트북에서 호출하기 위해 쓰인다.
import requests


In [45]:
# 로컬에 띄운 예측 서비스 주소. predict.py가 9696 포트에서 기다린다고 가정한다.
url = 'http://localhost:9696/predict'


In [51]:
# Flask에 보낼 고객 JSON. 계약이 two_year인 다른 시나리오로, API가 새 입력을 받는지 시험한다.
customer = {
    'gender': 'female',
    'seniorcitizen': 0,
    'partner': 'yes',
    'dependents': 'no',
    'phoneservice': 'no',
    'multiplelines': 'no_phone_service',
    'internetservice': 'dsl',
    'onlinesecurity': 'no',
    'onlinebackup': 'yes',
    'deviceprotection': 'no',
    'techsupport': 'no',
    'streamingtv': 'no',
    'streamingmovies': 'no',
    'contract': 'two_year',
    'paperlessbilling': 'yes',
    'paymentmethod': 'electronic_check',
    'tenure': 1,
    'monthlycharges': 29.85,
    'totalcharges': 29.85
}


In [52]:
# post()는 HTTP POST로 JSON을 보내는 함수다.
# .json()은 응답 본문을 파이썬 딕셔너리로 바꿔, churn 여부와 확률을 읽기 위해 쓰인다.
# 이 셀은 predict.py(또는 gunicorn)가 켜져 있어야 한다. 서버가 없으면 연결 에러가 난다.
response = requests.post(url, json=customer).json()


In [53]:
# API가 돌려 준 예측 결과를 확인한다.
response


{'churn': False, 'churn_probability': 0.29214211186746464}

In [55]:
# 이탈로 판단되면 마케팅 메일을 보낸다는 가짜 후속 처리다.
# 모델 점수를 실제 업무 결정에 어떻게 붙이는지 보여 주기 위해 쓰인다.
if response['churn']:
    print('sending email to', 'asdx-123d')
